# 01 - Setup and the classical answer

**Goal of this notebook:** get ViennaRNA working and produce the reference
answers we will try to reproduce with a quantum method.

**The problem.** An RNA molecule is a string of A, U, C, G. It folds back on
itself: A pairs with U, C pairs with G, and G can also pair weakly with U.
The set of pairs it forms is the *secondary structure*. Nature picks the
structure with the lowest free energy (the MFE structure).

**Dot-bracket notation.** `(` and `)` mark a pair, `.` marks an unpaired base.

**Honest framing, please read.** Classical dynamic programming solves this
exactly in O(L^3) time. There is no quantum speedup available here. We use
this problem because we *know* the right answer, so we can check whether our
quantum formulation works at all. The interesting extensions (pseudoknots,
multi-objective mRNA design) come at the end.

### How to run this notebook

This notebook runs on its own. You do not need the others open.

1. Put **gqe-rna.zip** at the top level of your Google Drive (My Drive).
   Do this once. You never need to unzip it yourself.
2. In Colab: **File -> Upload notebook** and pick this file.
3. **Runtime -> Change runtime type -> T4 GPU** (free tier is fine).
4. **Runtime -> Run all**. Approve the Drive popup when it appears.

Results are saved into `gqe-rna/results/` **in your Drive**, so they survive
when the runtime shuts down and the next notebook picks them up automatically.

If a cell says a file is missing, run the notebook it names first.

In [1]:
# STEP 1 - Connect to Drive and find the project.
# Put gqe-rna.zip at the top level of My Drive. Do not unzip it yourself.
import sys, os, glob, subprocess, importlib

from google.colab import drive
drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive"
REPO = os.path.join(ROOT, "gqe-rna")

# Unzip once, if the folder is not there yet.
if not os.path.isdir(os.path.join(REPO, "src")):
    zp = os.path.join(ROOT, "gqe-rna.zip")
    assert os.path.exists(zp), (
        "gqe-rna.zip was not found at the top level of My Drive. "
        "Upload it there (not in a subfolder), then run this cell again.")
    subprocess.run(["unzip", "-q", "-o", zp, "-d", ROOT], check=True)

# If the zip nested an extra folder, find the real one.
if not os.path.isdir(os.path.join(REPO, "src")):
    hits = glob.glob(os.path.join(ROOT, "**", "src", "colab_utils.py"),
                     recursive=True)
    assert hits, "Could not find src/ anywhere in My Drive after unzipping."
    REPO = os.path.dirname(os.path.dirname(hits[0]))

sys.path.insert(0, REPO)
importlib.invalidate_caches()   # Python cached this folder before it existed
from src import colab_utils as cu

cu.setup(repo=REPO)   # chdir into the project so results/ is saved to Drive
cu.install_vienna()

Mounted at /content/drive
project folder: /content/drive/MyDrive/gqe-rna
installing ViennaRNA ...


True

In [2]:
# STEP 2 - Load the project code.
import numpy as np
import matplotlib.pyplot as plt
from src import rna, energy, simulator, pools, metrics, baselines

rna.set_vienna_defaults()      # pin temperature and energy model
print("ViennaRNA available:", rna.HAVE_VIENNA)
cu.show_results()              # what earlier notebooks already saved

ViennaRNA available: True
file                        size   from notebook
--------------------------------------------------


## Our test sequences

The 44-nucleotide one is the example from the WISER/Moderna challenge PDF.
The shorter ones let us brute-force check everything.

In [3]:
SEQUENCES = {
    "challenge44": "GGAGCAAAACUUGUCGAUUGAGAACAAAAUACAGAAUUUGCUUG",
    "hairpin20":   "GGGAAAUCCCAAAGGGAUUU",
    "rand24":      rna.random_sequence(24, rng=1),
    "rand30":      rna.random_sequence(30, rng=2),
    "rand36":      rna.random_sequence(36, rng=3),
}
for k, v in SEQUENCES.items():
    print(f"{k:12s} L={len(v):3d}  {v}")

challenge44  L= 44  GGAGCAAAACUUGUCGAUUGAGAACAAAAUACAGAAUUUGCUUG
hairpin20    L= 20  GGGAAAUCCCAAAGGGAUUU
rand24       L= 24  GCACUUCUGACGUCUUAUAUCUUC
rand30       L= 30  UUCAGGAAUGGAUGUGCGUAUGCCUCUGAA
rand36       L= 36  AACGAUUAGAUGUGGCUGGUACUUCGUCAGUAGCAG


## The reference MFE structure

`RNA.fold` gives the structure and its energy in kcal/mol. This is the number
every result in this project is compared against.

In [4]:
# Fix the ViennaRNA settings so our numbers are reproducible.
# If you skip this, your energies may not match someone else's run.
rna.set_vienna_defaults(temperature=37.0, dangles=2)

refs = {}
for name, seq in SEQUENCES.items():
    db, e = rna.mfe_structure(seq)
    refs[name] = (db, e)
    print(f"{name:12s} {e:7.2f} kcal/mol")
    print(f"{'':12s} {seq}")
    print(f"{'':12s} {db}\n")

challenge44    -7.90 kcal/mol
             GGAGCAAAACUUGUCGAUUGAGAACAAAAUACAGAAUUUGCUUG
             .(((((((..((((...(((....)))...))))..))))))).

hairpin20      -6.40 kcal/mol
             GGGAAAUCCCAAAGGGAUUU
             ...(((((((...)))))))

rand24          0.00 kcal/mol
             GCACUUCUGACGUCUUAUAUCUUC
             ........................

rand30         -3.60 kcal/mol
             UUCAGGAAUGGAUGUGCGUAUGCCUCUGAA
             .(((((...(.(((....))).).))))).

rand36         -5.80 kcal/mol
             AACGAUUAGAUGUGGCUGGUACUUCGUCAGUAGCAG
             ...............(((.((((.....)))).)))



## Scoring any structure

We can also ask ViennaRNA for the energy of a structure *we* invented. This is
how we will grade the quantum results later: the energy gap to the MFE.

In [5]:
seq = SEQUENCES["challenge44"]
ref_db, ref_e = refs["challenge44"]

open_chain = "." * len(seq)
print("unfolded    :", rna.eval_structure(seq, open_chain), "kcal/mol")
print("MFE         :", round(ref_e, 2), "kcal/mol")

# A deliberately worse structure, to show the gap metric working.
worse = ref_db.replace("(", ".", 2).replace(")", ".", 2)
print("damaged MFE :", round(rna.eval_structure(seq, worse), 2), "kcal/mol")

unfolded    : 0.0 kcal/mol
MFE         : -7.9 kcal/mol
damaged MFE : 24.7 kcal/mol


## Suboptimal structures: what we are really after

`RNA.subopt` lists structures within some energy window of the MFE. A quantum
sampler gives a *distribution* over structures, not one answer, so this is the
fair thing to compare it to later.

In [6]:
if rna.HAVE_VIENNA:
    import RNA as V
    sub = V.subopt(SEQUENCES["hairpin20"], 200)   # delta is in decacal, so 200 = 2.0 kcal/mol
    print(f"{len(sub)} structures within 2 kcal/mol of the MFE")
    for s in list(sub)[:5]:
        print(f"  {s.energy:6.2f}  {s.structure}")

3 structures within 2 kcal/mol of the MFE
   -6.40  ...(((((((...)))))))
   -5.80  ....((((((...)))))).
   -4.90  .....(((((...)))))..


In [7]:
# Save the references for the other notebooks.
cu.save("refs", {"sequences": SEQUENCES, "refs": refs})

saved results/refs.pkl


'results/refs.pkl'

## What we learned

- ViennaRNA gives us exact reference structures and energies.
- Our grading metric will be **energy gap** and **base-pair F1**.
- Next: turn "which pairs form?" into a set of yes/no variables we can put on
  qubits. That is notebook 02.

In [8]:
# Everything saved so far. These files live in your Drive, so the next
# notebook will find them even after this runtime shuts down.
cu.show_results()

# Uncomment to download a copy to your computer:
# cu.download_results()

file                        size   from notebook
--------------------------------------------------
refs.pkl                    0.5K   01
